In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from lineval.utils import (load_hub_docs,
                           check_assign_docs,
                           OpenAiWordSenseComparatorv4,
                           ClusterByMeaningModelv4,
                           annotator_filter,
                           error_filter,
                           AnnotationSerializerv2,
                           AnnotationFactoryv2,
                           get_unique_senses
)
from linalgo.annotate.serializers import DocumentSerializer
from linalgo.annotate.models import Document

from linpub.metrics import accuracy

# models

In [12]:
load_dotenv()
api_key = os.getenv("OPEN_AI_API_KEY")
comp_1 = OpenAiWordSenseComparatorv4(api_key=api_key, openai_model="gpt-4o", thought_process=True)
model_1 = ClusterByMeaningModelv4(word_sense_comparator=comp_1)

# Getting data

In [6]:
X_docs = load_hub_docs()

Retrivieving task with id d3ce7764-eb85-4999-b965-c028f539ee33...
Retrieving annotators... (4 found)
Retrieving entities... (7 found)
Retrieving documents... (63 found)
Retrieving annotations... (431 found)


In [7]:
arnaud_id = 'e920598f-774d-4e1c-a2dc-8fb9a6f3053f'
jack_id = 'd34602e1-1664-42cb-b139-c5cb8bcfa2a0'
abi_id = 'eb9a2239-5a6c-4da5-8957-af8a90d3e880'
#Jack was used for creating the gold standard
X_gold = annotator_filter(X_docs, jack_id)
len(X_gold)

63

In [8]:
#checking if the number of annotations matches the number of contexts
all_annos = [ano for doc in X_gold for ano in list(doc.annotations)]
all_contexts = set([anno.body.context for doc in X_docs for anno in list(doc.annotations)])
len(all_annos), len(all_contexts)

(144, 144)

# Filtering data/errors if no gold

In [ ]:
er_fl_docs = error_filter(X_docs, keep_correct= True, keep_errors= False)

In [ ]:
#Don't run this, only if you want to send docs to Arnaud on the hub
# check_assign_docs(er_fl_docs)

# Predicting

In [13]:
#testing with only 3 docs
X_small = X_gold[:10]
y = get_unique_senses(X_small)
y_pred_docs = model_1.predict(X_small)
# y2 = get_unique_senses(X_small)
# y3 = get_unique_senses(X_gold[:10])
y_pred = get_unique_senses(y_pred_docs)
accuracy(y_pred, y) #, accuracy(y_pred, y2), accuracy(y_pred, y3)


            Compared se replier in 1. L'armée était en difficulté, il était préférable de se replier.
            with se replier in 1. L'armée était en difficulté, il était préférable de se replier.
            Proba: 1.0
            Process: reference word
            


NotFoundError: Error code: 404 - {'error': {'message': 'The model `4o` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [ ]:

y = get_unique_senses(X_gold)
y_pred_docs = model_1.predict(X_gold)
y_pred = get_unique_senses(y_pred_docs)
accuracy(y_pred, y)

# saving experiments

## experimenting

In [ ]:
X_docs = load_hub_docs()

In [ ]:
annotations = [ano for doc in X_docs for ano in list(doc.annotations) ]
lst = AnnotationSerializerv2(annotations).serialize()
len(lst), lst[0]

In [5]:
df = pd.DataFrame(lst)
df.to_csv("annotations.csv", index=False)

In [ ]:
df = pd.read_csv("annotations.csv")
lst = df.to_dict(orient="records")
len(lst)

In [4]:
rebuild_annotations = [AnnotationFactoryv2.from_dict(l) for l in lst]
# rebuild_annotations == annotations

In [ ]:
rebuild_annotations[0].document

In [9]:
docs_lst = DocumentSerializer(X_docs).serialize()
df = pd.DataFrame(docs_lst)
df.to_csv("docs.csv", index=False)


In [6]:

df = pd.read_csv("docs.csv")
lst = df.to_dict(orient="records")
rebuilt_docs = []
for l in lst:
    doc = Document(**l)
    rebuilt_docs.append(doc)

In [ ]:
rebuilt_docs[0].uri, rebuilt_docs[0].annotations[0].__dict__

In [ ]:
for doc in rebuilt_docs:
    doc_annos = []
    for anno in rebuild_annotations:
        if doc.uri == anno.document.uri:
            doc_annos.append(anno)
    doc.annotations = doc_annos

In [ ]:
rebuilt_docs == X_docs, rebuilt_docs[0]== X_docs[0], list(rebuilt_docs[0].annotations)[0] == list(X_docs[0].annotations)[0]

## save/load functions

In [6]:
def save_docs(docs, filename):
    annos = [ano for doc in docs for ano in list(doc.annotations)]
    anno_lst = AnnotationSerializerv2(annos).serialize()
    df = pd.DataFrame(anno_lst)
    df.to_csv(filename+"annos", index=False)
    docs_lst = DocumentSerializer(docs).serialize()
    df = pd.DataFrame(docs_lst)
    df.to_csv(filename+"docs", index=False)

save_docs(X_docs, "test")


In [3]:
def load_docs(filename):
    df = pd.read_csv(filename+"annos")
    lst = df.to_dict(orient="records")
    rebuild_annotations = [AnnotationFactoryv2.from_dict(l) for l in lst]
    df = pd.read_csv(filename+"docs")
    lst = df.to_dict(orient="records")
    rebuilt_docs = []
    for l in lst:
        doc = Document(**l)
        rebuilt_docs.append(doc)
    for doc in rebuilt_docs:
        doc_annos = []
        for anno in rebuild_annotations:
            if doc.uri == anno.document.uri:
                doc_annos.append(anno)
        doc.annotations = set(doc_annos)
    return rebuilt_docs

rebuilt_docs = load_docs("test")

In [4]:
list(rebuilt_docs[0].annotations)[0].__dict__

{'id': 'aedcbced-7599-45f1-a8e5-96c7313f733a',
 'entity': Entity::371a5d39-6e41-427f-8e1f-6bf64504e080,
 'score': None,
 'body': 'Body(text=\'se replier\', extras={\'context\': "1. L\'armée était en difficulté, il était préférable de se replier."})',
 'task': Task::d3ce7764-eb85-4999-b965-c028f539ee33,
 'annotator': Annotator::d34602e1-1664-42cb-b139-c5cb8bcfa2a0,
 'document': Document::04386177-799b-4a62-85f2-8a20e80559b2,
 'target': <linalgo.annotate.models.Target at 0x7f25f17fdd80>,
 'created': datetime.datetime(2024, 12, 27, 8, 10, 25, 258030, tzinfo=datetime.timezone.utc)}

In [5]:
rebuilt_docs[0].__dict__

{'id': '04386177-799b-4a62-85f2-8a20e80559b2',
 'uri': 62,
 'content': "Word: se replier\nId: 93eb335e-14e3-4407-a56b-c4c84017bd2a\nMeaning: 0\n------Context------\nJ’ai eu si mal au ventre, tout d’un coup, que je me suis repliée sur moi-même.\n-------------------\nWord: se replier\nId: 1571b9c0-8121-4893-afa5-8053def93237\nMeaning: 0\n------Context------\nL'armée était en difficulté, il était préférable de se replier.\n-------------------",
 'corpus': Corpus::default,
 'annotations': {Annotation::1ecf29a6-9193-4a8b-9317-5dc319b94507,
  Annotation::1ecf29a6-9193-4a8b-9317-5dc319b94507,
  Annotation::1ecf29a6-9193-4a8b-9317-5dc319b94507,
  Annotation::371a5d39-6e41-427f-8e1f-6bf64504e080,
  Annotation::371a5d39-6e41-427f-8e1f-6bf64504e080,
  Annotation::371a5d39-6e41-427f-8e1f-6bf64504e080}}

In [ ]:
list(X_docs[0].annotations)[0].__dict__

In [ ]:
X_docs[0].__dict__

# testing body serial/deserial

In [41]:
from lineval.other_stuff import Body, BodySerializer, BodyFactory

In [ ]:
Body1 = Body(text = "This is a test")
Body2 = Body1
Body2.proba = 1.0
Body2.process = "ref"
Body3 = Body2
Body3.text = "This is a test 2"
Body3.proba = 0.5
Body3.process = "plop"
Body3.context = "ref"
Body3.text = "This is a test 3"
Body4 = "not a Body class"
Body1, Body2, Body3, Body4

In [ ]:
ser1 = BodySerializer().serialize(Body1)
ser2 = BodySerializer().serialize(Body2)
ser3 = BodySerializer().serialize(Body3)
ser4 = BodySerializer().serialize(Body4)
ser1, ser2, ser3, ser4

In [ ]:
fact = BodyFactory()
res1 = fact.deserialize(ser1)
res2 = fact.deserialize(ser2)
res3 = fact.deserialize(ser3)
res4 = fact.deserialize(ser4)
res1, res2, res3, res4

In [ ]:
res1 == Body1, res2 == Body2, res3 == Body3, res4 == Body4

# testing ano serial/factory

In [3]:
from lineval.other_stuff import AnnotationSerializerv2, AnnotationFactoryv2

In [ ]:
from lineval.utils import load_hub_docs
X_docs = load_hub_docs()


In [ ]:
ano = list(X_docs[0].annotations)[0]
lst = AnnotationSerializerv2([ano]).serialize()
res = AnnotationFactoryv2.from_dict(lst[0])
res == ano

In [ ]:
annos = [ano for doc in X_docs for ano in list(doc.annotations)]
lst = AnnotationSerializerv2(annos).serialize()
rebuilt_annos = [AnnotationFactoryv2.from_dict(l) for l in lst]
len(rebuilt_annos), annos == rebuilt_annos

In [ ]:
rebuilt_docs = set([anno.document for anno in annos])
len(rebuilt_docs), len(X_docs)

In [ ]:
mismatch = False
for doc in X_docs:
    for r_doc in rebuilt_docs:
        if doc.uri == r_doc.uri and doc != r_doc:
            print(doc.uri)
            mismatch = True
if not mismatch:
    print("All good")
